# 🏥 Patient Health Deterioration Dashboard
### INT 375 Python Project — Krishika Malhotra (Reg. No. 12413944)
**Lovely Professional University | Supervisor: Dr. Savleen Kaur**

---
This interactive dashboard analyses 10,000 patients' hourly vital signs to detect early warning signs of clinical deterioration.

In [ ]:
# Install required libraries (uncomment if needed)
#!pip install hvplot panel

In [ ]:
import pandas as pd
import numpy as np
import panel as pn
import hvplot.pandas

pn.extension('tabulator')

print("Libraries loaded successfully ✅")

In [ ]:
# ─────────────────────────────────────────────
# Load & Cache Data
# ─────────────────────────────────────────────
if 'data' not in pn.state.cache.keys():
    df = pd.read_csv('hospital_deterioration_hourly_panel.csv')
    pn.state.cache['data'] = df.copy()
else:
    df = pn.state.cache['data']

print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Unique patients: {df['patient_id'].nunique():,}")
df.head(3)

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

## (0) Data Preprocessing

In [ ]:
# Fill missing values with 0
df = df.fillna(0)

# Derived columns
df['pulse_pressure'] = df['systolic_bp'] - df['diastolic_bp']
df['hr_spo2_ratio']  = df['heart_rate'] / df['spo2_pct'].replace(0, np.nan)

# Make interactive pipeline
idf = df.interactive()

print("Preprocessing complete ✅")

## (1) Panel Widgets — Global Controls

In [ ]:
# ── Hour slider ──────────────────────────────
hour_slider = pn.widgets.IntSlider(
    name='Hour from Admission',
    start=0, end=int(df['hour_from_admission'].max()),
    step=1, value=12
)

# ── Admission type multi-select ───────────────
admission_type_select = pn.widgets.CheckBoxGroup(
    name='Admission Type',
    options=['Elective', 'ED', 'Transfer'],
    value=['Elective', 'ED', 'Transfer']
)

# ── Vital sign radio buttons ──────────────────
yaxis_vital = pn.widgets.RadioButtonGroup(
    name='Vital Sign (time-series)',
    options=['heart_rate', 'respiratory_rate', 'spo2_pct',
             'temperature_c', 'systolic_bp', 'sepsis_risk_score'],
    button_type='success'
)

# ── Lab marker radio buttons ──────────────────
yaxis_lab = pn.widgets.RadioButtonGroup(
    name='Lab Marker (bar chart)',
    options=['wbc_count', 'lactate', 'creatinine', 'crp_level', 'hemoglobin'],
    button_type='warning'
)

print("Widgets created ✅")

## (2) Plot 1 — Vital Sign Trend Over Time (by Deterioration Status)

In [ ]:
vital_trend_pipeline = (
    idf[
        (idf.hour_from_admission <= hour_slider) &
        (idf.admission_type.isin(admission_type_select))
    ]
    .groupby(['hour_from_admission', 'deterioration_event'])[yaxis_vital].mean()
    .to_frame()
    .reset_index()
    .sort_values('hour_from_admission')
    .reset_index(drop=True)
)

vital_trend_pipeline

In [ ]:
vital_trend_plot = vital_trend_pipeline.hvplot(
    x='hour_from_admission',
    y=yaxis_vital,
    by='deterioration_event',
    line_width=2,
    title='Vital Sign Trend: Deteriorated vs Non-Deteriorated Patients',
    xlabel='Hour from Admission',
    height=350, width=700
)
vital_trend_plot

## (3) Plot 2 — Tabulator: Patient-Level Summary at Selected Hour

In [ ]:
summary_pipeline = (
    idf[
        (idf.hour_from_admission == hour_slider) &
        (idf.admission_type.isin(admission_type_select))
    ]
    [['patient_id', 'heart_rate', 'respiratory_rate', 'spo2_pct',
      'temperature_c', 'systolic_bp', 'sepsis_risk_score',
      'deterioration_event', 'admission_type', 'gender', 'age']]
    .reset_index(drop=True)
)

patient_table = summary_pipeline.pipe(
    pn.widgets.Tabulator,
    pagination='remote',
    page_size=10,
    sizing_mode='stretch_width'
)
patient_table

## (4) Plot 3 — Sepsis Risk Score vs SpO₂ Scatter (per Patient)

In [ ]:
scatter_pipeline = (
    idf[
        (idf.hour_from_admission == hour_slider) &
        (idf.admission_type.isin(admission_type_select))
    ]
    .groupby(['patient_id', 'gender'])['sepsis_risk_score', 'spo2_pct',
                                       'heart_rate', 'deterioration_event'].mean()
    .to_frame()
    .reset_index()
)

scatter_pipeline

In [ ]:
scatter_plot = scatter_pipeline.hvplot(
    x='spo2_pct',
    y='sepsis_risk_score',
    by='gender',
    kind='scatter',
    size=40, alpha=0.5,
    title='Sepsis Risk Score vs SpO₂ at Selected Hour',
    xlabel='SpO₂ (%)', ylabel='Sepsis Risk Score',
    legend=True, height=400, width=500
)
scatter_plot

## (5) Plot 4 — Lab Marker Bar Chart by Admission Type

In [ ]:
lab_bar_pipeline = (
    idf[
        (idf.hour_from_admission == hour_slider) &
        (idf.admission_type.isin(admission_type_select))
    ]
    .groupby(['admission_type', 'deterioration_event'])[yaxis_lab].mean()
    .to_frame()
    .reset_index()
    .reset_index(drop=True)
)

lab_bar_pipeline

In [ ]:
lab_bar_plot = lab_bar_pipeline.hvplot(
    kind='bar',
    x='admission_type',
    y=yaxis_lab,
    by='deterioration_event',
    title='Lab Marker by Admission Type & Deterioration Status',
    xlabel='Admission Type',
    height=400, width=550
)
lab_bar_plot

## (6) Assemble Full Dashboard

In [ ]:
template = pn.template.FastListTemplate(
    title='🏥 Patient Health Deterioration Dashboard',

    # ── Sidebar ───────────────────────────────
    sidebar=[
        pn.pane.Markdown("# 🏥 Patient Deterioration\nINT 375 Project — Krishika Malhotra"),
        pn.pane.Markdown(
            """This dashboard analyses **10,000 hospital patients** 
            monitored hourly for physiological deterioration.\n\n
            Use the controls below to filter the data."""
        ),
        pn.pane.Markdown("## ⚙️ Global Filters"),
        hour_slider,
        pn.pane.Markdown("### Admission Type"),
        admission_type_select,
    ],

    # ── Main content ──────────────────────────
    main=[
        # Row 1: Time-series + Table
        pn.Row(
            pn.Column(
                pn.pane.Markdown("### Vital Sign Over Time"),
                yaxis_vital,
                vital_trend_plot.panel(width=700),
                margin=(0, 25)
            ),
            pn.Column(
                pn.pane.Markdown("### Patient Snapshot at Selected Hour"),
                patient_table.panel(width=500)
            )
        ),

        # Row 2: Scatter + Bar chart
        pn.Row(
            pn.Column(
                scatter_plot.panel(width=550),
                margin=(0, 25)
            ),
            pn.Column(
                pn.pane.Markdown("### Lab Marker by Admission Type"),
                yaxis_lab,
                lab_bar_plot.panel(width=600)
            )
        ),
    ],

    # ── Theme ─────────────────────────────────
    accent_base_color="#2196F3",
    header_background="#1565C0",
)

template.show()      # Opens in a new browser tab
template.servable()  # Makes it servable via `panel serve notebook.ipynb`